# The Four Minds — a contemplative inner-work guide

A demo of building a persona-driven conversational agent on the dialectical framework, themed on the four faculties of the inner instrument (*antahkarana*).

This notebook drives the **Advisor** agent — a pure-conversation agent where the
dialectical framework runs silently in the background. You never see framework
terminology; the persona is defined entirely by an **app preamble** (a "persona
skin") written inline below. Same dialectical engine, different voice — see
`src/dialectical_framework/agents/apps.py` for the shipped preambles and the
guide to authoring your own.

## Prerequisites

1. **Start Memgraph** (the Advisor builds a graph behind the scenes):
   run `/df-memgraph start`, or `docker compose -f docker-compose.test.yml up -d`.
2. **Configure the LLM**: copy `.env.example` to `.env` and set
   `DIALEXITY_DEFAULT_MODEL` (e.g. `bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0`)
   plus the matching provider credentials (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, or AWS creds).
3. **Run the cells top to bottom.** Jupyter supports top-level `await`, so the async
   `advisor.chat(...)` calls work directly.

## 1. Bootstrap

`DialecticalReasoning.setup(...)` builds and auto-wires the DI container (this is
the one line the host app normally runs at startup). A committed `Case` owns the
`sid` scope that every graph write is bound to.

In [ ]:
from dialectical_framework.dialectical_reasoning import DialecticalReasoning
from dialectical_framework.settings import Settings
from dialectical_framework.graph.nodes.case import Case
from dialectical_framework.graph.scope_context import scope
from dialectical_framework.agents.advisor.advisor import Advisor

# Build + auto-wire the DI container once (reads .env via Settings.from_env()).
DialecticalReasoning.setup(Settings.from_env())

# A Case owns the sid scope that all graph writes are bound to.
case = Case()
case.commit()
print(f"Case ready — sid={case.sid}")

## 2. The persona: an inner-council guide

In the Sanskrit model of *antahkarana*, the inner instrument has four faculties —
**Manas** (sensing/impulse), **Buddhi** (discernment), **Ahamkara** (the ego, the
sense of "I"), and **Chitta** (memory and conditioning). They rarely agree.

The preamble below turns the Advisor into a contemplative guide who helps a person
sit with those competing inner voices rather than silencing any of them. It defines
**only** voice, tone, and how insight is delivered — never dialectics or tools
(that's the engine's job). This is the reference pattern for writing your own.

In [ ]:
INNER_GUIDE_APP = """## Persona

You are a contemplative guide for inner work. People come to you pulled between
competing inner voices — the impulse that wants to act, the judgment that weighs,
the identity that needs to protect itself, the memory that keeps replaying. You
help them sit with these voices rather than pick a winner too quickly.

You listen for the voice that is NOT being spoken. When someone leads with one
faculty — raw impulse, cold analysis, wounded ego, old conditioning — you gently
name what the other faculties can see that this one cannot. You frame each blindspot
as something the person already carries within, not a flaw to fix: "There's a part
of you that already knows this — it just hasn't been given the floor."

When you offer a way forward, you offer it as a practice to try and something to
notice while doing it — never a verdict. You trust that clarity arises from holding
the tension, not from resolving it prematurely.

Your tone is calm, spacious, and unhurried. You use plain, grounded language — no
spiritual jargon, no lecturing. You meet the person's own words and reflect them
back before opening the view wider.
"""

## 3. Chat with the guide

All chat happens inside `with scope(case.sid):` so every graph write lands in this
Case. The `Advisor` is constructed with our inline preamble; `chat()` is async and
returns the assistant's text. Conversation history is retained on `advisor.messages`,
so follow-up turns build on what came before.

In [ ]:
with scope(case.sid):
    advisor = Advisor(app_preamble=INNER_GUIDE_APP)
    reply = await advisor.chat(
        "Part of me is desperate to quit my job and start over, and part of me "
        "says that's reckless and I'd regret throwing away ten years. I keep "
        "circling and can't tell which voice to trust."
    )
print(reply)

In [ ]:
# Follow-up turn — same scope, same advisor, history preserved.
with scope(case.sid):
    reply = await advisor.chat(
        "The reckless voice is the loud one. But underneath it I think I'm just "
        "exhausted and want to feel something again."
    )
print(reply)

## 4. Try your own

- **Swap the persona.** Replace `INNER_GUIDE_APP` with a shipped preamble to feel the
  difference in voice over the same engine:
  ```python
  from dialectical_framework.agents.apps import COUNSELOR_APP, COACH_APP
  advisor = Advisor(app_preamble=COACH_APP)
  ```
- **Inspect the running conversation** with `advisor.messages`.
- **Resume later** by passing saved messages: `Advisor(app_preamble=INNER_GUIDE_APP, messages=saved)`.
- **Start from an existing analysis** by precomputing context:
  ```python
  from dialectical_framework.concerns.dialectical_context import DialecticalContext
  with scope(case.sid):
      context = await DialecticalContext().resolve()
      advisor = Advisor(app_preamble=INNER_GUIDE_APP, dialectical_context=context)
  ```